<a href="https://colab.research.google.com/github/anasmita3/SURGE/blob/main/SURGE_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install datasets

In [ ]:
from huggingface_hub import login

login("")

In [ ]:
from datasets import load_dataset

ds = load_dataset("Vacaspati/Vacaspati")

print(ds)

print("\nColumns:")
print(ds["train"].column_names)

print("\nSample:")
print(ds["train"][0])

In [ ]:
import unicodedata
import re

count = 0

with open("bn_literature.txt", "w", encoding="utf-8") as out:

    for row in ds["train"]:

        text = row["text"]

        text = unicodedata.normalize("NFC", text)
        text = re.sub(r"\s+", " ", text).strip()

        if not text:
            continue

        out.write(text + "\n")
        count += 1

print("Saved documents:", count)

In [ ]:
import random

random.seed(42)

TARGET = 125000

with open("bn_literature.txt", encoding="utf-8") as f:
    total_docs = sum(1 for _ in f)

print("Total docs:", total_docs)

selected = set(
    random.sample(
        range(total_docs),
        TARGET
    )
)

count = 0

with open("bn_literature.txt", encoding="utf-8") as inp, \
     open("literature_125k.txt", "w", encoding="utf-8") as out:

    for idx, line in enumerate(inp):

        if idx in selected:
            out.write(line)
            count += 1

print("Saved:", count)

In [ ]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    "ai4bharat/IndicCorpV2",
    repo_type="dataset"
)

print(len(files))
print(files[:20])

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "ai4bharat/IndicCorpV2",
    data_files={"train": "data/bn.txt"}
)

print(dataset)

In [ ]:
import unicodedata
import re

MAX_DOCS = 500000

with open("bn_news.txt", "w", encoding="utf-8") as f:

    for i, row in enumerate(dataset["train"]):

        text = unicodedata.normalize("NFC", row["text"])
        text = re.sub(r"\s+", " ", text).strip()

        if text:
            f.write(text + "\n")

        if i + 1 >= MAX_DOCS:
            break

print("Saved", MAX_DOCS, "documents")

In [ ]:
import kagglehub

path = kagglehub.dataset_download(
    "mdsalmanhossain/bangla-social-media-abuse-dataset"
)

print(path)

In [ ]:
import os

for root, dirs, files in os.walk(path):
    for f in files:
        print(f)

In [ ]:
import pandas as pd
import os

file_path = os.path.join(
    path,
    "cyberbulling bangla dataset.xlsx"
)

xls = pd.ExcelFile(file_path)

print(xls.sheet_names)
df = pd.read_excel(
    file_path,
    sheet_name=0
)

print(df.columns.tolist())
print()
print("Rows:", len(df))

In [ ]:
import os

for f in os.listdir():
    if f.endswith(".txt"):
        print(f)

In [ ]:
import random

random.seed(42)

TARGET = 125000

with open("bn_news.txt", encoding="utf-8") as f:
    total_docs = sum(1 for _ in f)

print("Total news docs:", total_docs)

selected = set(
    random.sample(
        range(total_docs),
        TARGET
    )
)

count = 0

with open("bn_news.txt", encoding="utf-8") as inp, \
     open("news_125k.txt", "w", encoding="utf-8") as out:

    for idx, line in enumerate(inp):

        if idx in selected:
            out.write(line)
            count += 1

print("Saved:", count)

In [ ]:
print(df.columns.tolist())
print("Rows:", len(df))

In [ ]:
social_docs = [
    str(x).strip()
    for x in df["comment"]
    if pd.notna(x) and str(x).strip()
]

print("Social docs:", len(social_docs))

In [ ]:
with open(
    "social_44k.txt",
    "w",
    encoding="utf-8"
) as f:

    for doc in social_docs:
        f.write(doc + "\n")

print("Saved:", len(social_docs))

In [ ]:
social_docs = [
    str(x).strip()
    for x in df["comment"]
    if pd.notna(x) and str(x).strip()
]

print("Original social:", len(social_docs))

In [ ]:
social_125k = []

while len(social_125k) < 125000:
    social_125k.extend(social_docs)

social_125k = social_125k[:125000]

print("Expanded social:", len(social_125k))

In [ ]:
with open("social_125k.txt", "w", encoding="utf-8") as f:
    for doc in social_125k:
        f.write(doc + "\n")

In [ ]:
with open("news_125k.txt", encoding="utf-8") as f:
    count = sum(1 for _ in f)

print("News documents:", count)

In [ ]:
with open("literature_125k.txt", encoding="utf-8") as f:
    literature_docs = [x.strip() for x in f if x.strip()]

with open("news_125k.txt", encoding="utf-8") as f:
    news_docs = [x.strip() for x in f if x.strip()]

all_docs = (
    literature_docs +
    news_docs +
    social_125k
)

import random
random.seed(42)
random.shuffle(all_docs)

print("Total documents:", len(all_docs))

In [ ]:
with open(
    "all_domains_375k.txt",
    "w",
    encoding="utf-8"
) as f:

    for doc in all_docs:
        f.write(doc.strip() + "\n")

print("Saved:", len(all_docs))

In [ ]:
from collections import Counter

chars = 0
words = 0
vocab = Counter()

for doc in all_docs:

    chars += len(doc)

    ws = doc.split()

    words += len(ws)

    vocab.update(ws)

print("Documents :", len(all_docs))
print("Characters:", chars)
print("Words     :", words)
print("Vocabulary:", len(vocab))
print("TTR       :", len(vocab)/words)

avg_len = sum(len(w) for w in vocab) / len(vocab)

print("Average Word Length:", avg_len)

In [ ]:
from sklearn.model_selection import train_test_split

train_docs, test_docs = train_test_split(
    all_docs,
    test_size=0.10,
    random_state=42
)

print("Train:", len(train_docs))
print("Test :", len(test_docs))

In [ ]:
with open("train_all_bn.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(train_docs))

with open("test_all_bn.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(test_docs))

Transliteration

In [ ]:
!pip install -q aksharamukha

from aksharamukha import transliterate
from tqdm import tqdm

def convert_file(inp, out):

    with open(inp, encoding="utf-8") as f:
        lines = [x.rstrip("\n") for x in f]

    with open(out, "w", encoding="utf-8") as f:

        for line in tqdm(lines):

            try:
                ascii_text = transliterate.process(
                    "Bengali",
                    "HK",          # ASCII transliteration
                    line
                )

            except Exception:
                ascii_text = line

            f.write(ascii_text + "\n")

    print("Saved:", out)

In [ ]:
convert_file(
    "train_all_bn.txt",
    "train_all_iso.txt"
)

convert_file(
    "test_all_bn.txt",
    "test_all_iso.txt"
)

In [ ]:
from aksharamukha import transliterate
import random

with open(
    "test_all_bn.txt",
    encoding="utf-8"
) as f:

    samples = [
        x.strip()
        for x in f
        if x.strip()
    ]

samples = random.sample(samples, 100)

correct = 0

for s in samples:

    iso = transliterate.process(
        "Bengali",
        "ISO",
        s
    )

    back = transliterate.process(
        "ISO",
        "Bengali",
        iso
    )

    if s == back:
        correct += 1

print(
    "Exact Match:",
    correct,
    "/ 100"
)

print(
    "Accuracy:",
    correct / 100
)

In [ ]:
for s in random.sample(samples, 10):

    iso = transliterate.process(
        "Bengali",
        "ISO",
        s
    )

    back = transliterate.process(
        "ISO",
        "Bengali",
        iso
    )

    print("="*80)
    print("ORIGINAL:")
    print(s)

    print("\nBACK:")
    print(back)

In [ ]:
from collections import Counter

def corpus_stats(path):

    chars = 0
    words = 0
    vocab = Counter()

    with open(path, encoding="utf-8") as f:

        for line in f:

            chars += len(line)

            ws = line.split()

            words += len(ws)

            vocab.update(ws)

    return {
        "chars": chars,
        "words": words,
        "vocab": len(vocab),
        "avg_word_len": chars / words
    }

bn = corpus_stats(
    "train_all_bn.txt"
)

iso = corpus_stats(
    "train_all_iso.txt"
)

print("Bengali:", bn)
print("ISO:", iso)

print(
    "\nExpansion Factor:",
    iso["chars"] / bn["chars"]
)

print(
    "Vocabulary Ratio:",
    iso["vocab"] / bn["vocab"]
)

Tokenization

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

In [ ]:
VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

In [ ]:
for vocab_size in VOCABS:

    tokenizer = Tokenizer(
        BPE(unk_token="[UNK]")
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"]
    )

    tokenizer.train(
        ["train_all_bn.txt"],
        trainer
    )

    tokenizer.save(
        f"bpe_bn_{vocab_size}.json"
    )

    print(
        f"Finished {vocab_size}"
    )

In [ ]:
from tokenizers import Tokenizer
import numpy as np

def evaluate_tokenizer(tokenizer_path, test_file):

    tok = Tokenizer.from_file(tokenizer_path)

    total_words = 0
    total_tokens = 0
    total_chars = 0

    unk_count = 0

    fertilities = []
    token_lengths = []

    with open(test_file, encoding="utf-8") as f:

        for line in f:

            ws = line.strip().split()

            for w in ws:

                total_words += 1
                total_chars += len(w)

                enc = tok.encode(w)

                tks = enc.tokens

                n_tok = len(tks)

                total_tokens += n_tok

                fertilities.append(n_tok)

                for t in tks:
                    token_lengths.append(len(t))

                unk_count += tks.count("[UNK]")

    fertility = total_tokens / total_words

    avg_token_len = np.mean(token_lengths)

    cpt = total_chars / total_tokens

    compression = cpt

    avg_tokens = fertility

    wfr = (total_tokens - total_words) / total_words

    variance = np.var(fertilities)

    return {
        "oov": unk_count / total_words,
        "fertility": fertility,
        "cpt": cpt,
        "compression": compression,
        "avg_tokens": avg_tokens,
        "wfr": wfr,
        "variance": variance
    }

In [ ]:
results = []

for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"bpe_bn_{vocab_size}.json",
        "test_all_bn.txt"
    )

    r["vocab"] = vocab_size

    results.append(r)

    print(r)

print("\nFinished.")

In [ ]:
for vocab_size in VOCABS:

    tokenizer = Tokenizer(
        BPE(unk_token="[UNK]")
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"]
    )

    tokenizer.train(
        ["train_all_iso.txt"],
        trainer
    )

    tokenizer.save(
        f"bpe_iso_{vocab_size}.json"
    )

    print(vocab_size)

In [ ]:
results = []

for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"bpe_iso_{vocab_size}.json",
        "test_all_iso.txt"
    )

    r["vocab"] = vocab_size

    results.append(r)

    print(r)

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    tok = Tokenizer(
        WordPiece(
            unk_token="[UNK]"
        )
    )

    tok.pre_tokenizer = Whitespace()

    trainer = WordPieceTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"]
    )

    tok.train(
        ["train_all_bn.txt"],
        trainer
    )

    tok.save(
        f"wp_bn_{vocab_size}.json"
    )

    print(vocab_size)

In [ ]:
results = []

for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"wp_bn_{vocab_size}.json",
        "test_all_bn.txt"
    )

    r["vocab"] = vocab_size

    results.append(r)

    print(r)

print("\nFinished.")

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

for vocab_size in VOCABS:

    tok = Tokenizer(
        WordPiece(
            unk_token="[UNK]"
        )
    )

    tok.pre_tokenizer = Whitespace()

    trainer = WordPieceTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"]
    )

    tok.train(
        ["train_all_iso.txt"],
        trainer
    )

    tok.save(
        f"wp_iso_{vocab_size}.json"
    )

    print(vocab_size)

In [ ]:
results = []

for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"wp_iso_{vocab_size}.json",
        "test_all_iso.txt"
    )

    r["vocab"] = vocab_size

    results.append(r)

    print(r)

print("\nFinished.")

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import Unigram
from tokenizers.trainers import UnigramTrainer
from tokenizers.pre_tokenizers import Whitespace

for vocab_size in VOCABS:

    tok = Tokenizer(
        Unigram()
    )

    tok.pre_tokenizer = Whitespace()

    trainer = UnigramTrainer(
        vocab_size=vocab_size,
        unk_token="[UNK]",
        special_tokens=["[UNK]"]
    )

    tok.train(
        ["train_all_bn.txt"],
        trainer
    )

    tok.save(
        f"uni_bn_{vocab_size}.json"
    )

    print(vocab_size)

In [ ]:
results = []

for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"uni_bn_{vocab_size}.json",
        "test_all_bn.txt"
    )

    r["vocab"] = vocab_size

    results.append(r)

    print(r)

print("\nFinished.")

In [ ]:
for vocab_size in VOCABS:

    tok = Tokenizer(
        Unigram()
    )

    tok.pre_tokenizer = Whitespace()

    trainer = UnigramTrainer(
        vocab_size=vocab_size,
        unk_token="[UNK]",
        special_tokens=["[UNK]"]
    )

    tok.train(
        ["train_all_iso.txt"],
        trainer
    )

    tok.save(
        f"uni_iso_{vocab_size}.json"
    )

    print(vocab_size)

In [ ]:
results = []

for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"uni_iso_{vocab_size}.json",
        "test_all_iso.txt"
    )

    r["vocab"] = vocab_size

    results.append(r)

    print(r)

print("\nFinished.")

In [ ]:
import matplotlib.pyplot as plt

vocab = [5000,10000,15000,20000,25000,30000,35000,40000,45000,50000]

bpe = [1.812,1.568,1.472,1.418,1.384,1.359,1.340,1.326,1.314,1.305]
wp  = [2.003,1.651,1.529,1.461,1.418,1.388,1.366,1.348,1.334,1.323]
uni = [1.807,1.578,1.493,1.446,1.419,1.400,1.386,1.376,1.368,1.362]

plt.figure(figsize=(8,5))
plt.plot(vocab,bpe,marker='o',label='BPE')
plt.plot(vocab,wp,marker='s',label='WordPiece')
plt.plot(vocab,uni,marker='^',label='Unigram')

plt.xlabel("Vocabulary Size")
plt.ylabel("Fertility")
plt.title("Bengali Script: Fertility vs Vocabulary Size")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

vocab=[5000,10000,15000,20000,25000,30000,35000,40000,45000,50000]

bpe=[3.484,4.005,4.256,4.411,4.515,4.593,4.653,4.700,4.738,4.770]
wp=[3.151,3.792,4.090,4.274,4.400,4.492,4.564,4.621,4.666,4.704]
uni=[3.414,3.902,4.123,4.250,4.333,4.389,4.431,4.461,4.486,4.507]

plt.figure(figsize=(8,5))
plt.plot(vocab,bpe,marker='o',label='BPE')
plt.plot(vocab,wp,marker='s',label='WordPiece')
plt.plot(vocab,uni,marker='^',label='Unigram')

plt.xlabel("Vocabulary Size")
plt.ylabel("Characters per Token")
plt.title("Ascii Script: Compression Efficiency")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

vocab = [5000,10000,15000,20000,25000,30000,35000,40000,45000,50000]

bpe_bn = [2.925,3.379,3.599,3.737,3.829,3.899,3.954,3.997,4.031,4.060]
bpe_iso = [3.484,4.005,4.256,4.411,4.515,4.593,4.653,4.700,4.738,4.770]

plt.figure(figsize=(8,5))

plt.plot(
    vocab,
    bpe_bn,
    marker='o',
    linewidth=2,
    label='Bengali Script'
)

plt.plot(
    vocab,
    bpe_iso,
    marker='s',
    linewidth=2,
    label='ASCII Script'
)

plt.xlabel('Vocabulary Size')
plt.ylabel('Characters per Token (CPT)')
plt.title('BPE Compression Efficiency: Bengali vs ASCII')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

tokenizers = ['BPE', 'WordPiece', 'Unigram']

bn = [4.060, 4.004, 3.891]
iso = [4.770, 4.704, 4.507]

x = np.arange(len(tokenizers))
width = 0.35

plt.figure(figsize=(8,5))

plt.bar(
    x - width/2,
    bn,
    width,
    label='Bengali'
)

plt.bar(
    x + width/2,
    iso,
    width,
    label='ASCII'
)

plt.xticks(x, tokenizers)
plt.ylabel('Characters per Token (CPT)')
plt.title('Final Tokenizer Comparison at 50K Vocabulary')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
pip install nbformat

| Tokenizer | Script |          OOV |  Fertility |        WFR |   Variance |
| --------- | ------ | -----------: | ---------: | ---------: | ---------: |
| BPE       | Bangla |     0.000083 | **1.3050** | **0.3050** | **0.4939** |
| BPE       | ASCII  |     0.000083 |     1.3132 |     0.3132 |     0.5002 |
| WordPiece | Bangla |     0.000033 |     1.3232 |     0.3232 |     0.5438 |
| WordPiece | ASCII  |     0.000033 |     1.3321 |     0.3321 |     0.5551 |
| Unigram   | Bangla | **0.000000** |     1.3618 |     0.3618 |     0.5645 |
| Unigram   | ASCII  | **0.000000** |     1.3848 |     0.3848 |     0.6100 |
